# TV3 — Random Forest (scikit-learn, KHÔNG dùng Spark)

Dùng cùng 10 đặc trưng và cùng bộ dữ liệu train/val như Logistic Regression để so sánh công bằng. **Không đọc tập Test trong notebook này.**

In [ ]:
# CELL 1 — Cài thư viện (không cài Spark)
%pip install -q "numpy>=1.26,<3" "pandas>=2.1,<3" "pyarrow>=14,<24" "scikit-learn>=1.4,<2" "matplotlib>=3.7,<4" "joblib>=1.3,<2"


In [ ]:
# CELL 2 — Kết nối Drive và chọn đường dẫn (SỬA các biến sau cho phù hợp)
import sys, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Repo: thư mục chứa src/model_lr.py, src/model_rf.py, notebooks/... đã tải từ GitHub
REPO = Path('/content/BDA-Nhom_8-main')
# Drive của TV2: thư mục có train/, val/, test/ và manifest.json
FEATURES_ON_DRIVE = Path('/content/drive/MyDrive/BigData_Topic20/01_data_frozen/features_v1')
# Kết quả lưu trên Drive, tránh mất dữ liệu khi Colab ngắt phiên
OUTPUT_ROOT = Path('/content/drive/MyDrive/BigData_Topic20/02_models/tv3_sklearn')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO / 'src' / 'tv3_common.py').is_file():
    raise FileNotFoundError(f'Thiếu code TV3 tại {REPO}. Hãy upload/clone repo đã ghép file TV3.')
if not (FEATURES_ON_DRIVE / 'train').is_dir():
    raise FileNotFoundError(f'Chưa thấy bảng feature TV2: {FEATURES_ON_DRIVE}')

# Copy dữ liệu sang đĩa cục bộ Colab cho tốc độ đọc tốt hơn.
FEATURES_DIR = Path('/content/features_v1')
if not (FEATURES_DIR / 'train').is_dir():
    shutil.copytree(FEATURES_ON_DRIVE, FEATURES_DIR, dirs_exist_ok=True)
print('REPO:', REPO)
print('FEATURES_DIR:', FEATURES_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
sys.path.insert(0, str(REPO))


In [ ]:
# CELL 3 — Đọc dữ liệu cố định và kiểm tra số dòng/schema
from src.tv3_common import read_split, CAT_COLS, NUM_COLS, FEATURE_COLS, assert_disjoint
train = read_split(FEATURES_DIR, 'train')
val = read_split(FEATURES_DIR, 'val')
assert_disjoint(train, val)
print('Train:', train.shape, 'Validation:', val.shape)
print('Biến phân loại:', CAT_COLS)
print('Biến số:', NUM_COLS)
print('Tỷ lệ trễ Train:', round(train['Delay'].mean(), 4))
display(train.head(5))


## CELL 4 — Huấn luyện Random Forest với các bộ siêu tham số

Thử `n_estimators`, `max_depth`, `min_samples_leaf`. Chọn cấu hình bằng Validation, tuyệt đối không chọn bằng Test.

In [ ]:
from src.model_rf import train_rf
rf_metrics = train_rf(FEATURES_DIR, OUTPUT_ROOT)
print('Validation metrics RF:')
for name, val in rf_metrics.items():
    print(name, ':', val)

## CELL 5 — Xem bảng thử nghiệm và độ quan trọng đặc trưng

In [ ]:
import pandas as pd
from IPython.display import display, Image
res = OUTPUT_ROOT / 'results'
display(pd.read_csv(res / 'rf_experiments.csv').sort_values(['roc_auc', 'pr_auc'], ascending=False))
display(pd.read_csv(res / 'rf_feature_importances.csv').head(20))
for plot in ['rf_roc.png', 'rf_pr.png', 'rf_confusion.png']:
    display(Image(filename=str(OUTPUT_ROOT / 'reports' / 'tv3' / plot)))

## CELL 6 — So sánh LR vs RF trên Validation

In [ ]:
import json
rows = []
for name in ['lr', 'rf']:
    f = res / f'val_metrics_{name}.json'
    if f.exists():
        m = json.loads(f.read_text(encoding='utf-8'))
        rows.append({'model':name, **{k:v for k,v in m.items() if k != 'confusion_matrix'}})
if not rows:
    print('Chưa có model nào được huấn luyện')
else:
    display(pd.DataFrame(rows)[['model','roc_auc','pr_auc','accuracy','precision','recall','f1','threshold']])

## Hoàn thành RF
Đã lưu `models/rf/best_model.joblib`, cấu hình, bảng thử nghiệm và dự đoán Validation. Sau khi nhóm chốt các cấu hình, mở notebook `06_test_tv3.ipynb` để chạy Test đúng một lần.